# 보조 런타임 (여러 런타임 동시 실행)

본 노트북과 같은 실험 폴더에서 남은 작업을 나눠 맡습니다. 1번 → 3번 → 4-B번 순서로 실행하세요. 런타임 유형은 CPU 코어가 많을수록 좋습니다(예: TPU v2-8은 CPU 96코어).


In [ ]:
# 1. 검증된 GitHub 소스 버전 가져오기 및 설치
import os, sys, subprocess, json
from pathlib import Path
REPOSITORY = 'https://github.com/phj9187/foodcourt-adaptive-sim.git'
SOURCE_COMMIT = '4355d5299094a0dde39e13a1cdda972a384b3884'  # 642cf41 + kiosk input time + named tastes
WORK = Path('/content/foodcourt-adaptive-sim')
if not WORK.exists():
    subprocess.run(['git','clone',REPOSITORY,str(WORK)],check=True)
else:
    origin = subprocess.check_output(['git','-C',str(WORK),'remote','get-url','origin'],text=True).strip()
    if origin != REPOSITORY:
        raise RuntimeError('같은 경로에 다른 프로젝트가 있습니다. 새 런타임을 사용하세요.')
    subprocess.run(['git','-C',str(WORK),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(WORK),'checkout','--detach',SOURCE_COMMIT],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(WORK)],check=True)
# An editable install is only seen by new interpreters; make it importable in this kernel too.
if str(WORK/'src') not in sys.path:
    sys.path.insert(0,str(WORK/'src'))
print('소스 버전:',subprocess.check_output(['git','-C',str(WORK),'rev-parse','HEAD'],text=True).strip())
# Multi-runtime runner (scheduling only; does not change the manifest fingerprint)
TOOLS_COMMIT = 'd38f678750dc41b3a9ea6086ac2db692185443b8'
subprocess.run(['git','-C',str(WORK),'fetch','-q','origin',TOOLS_COMMIT],check=False)
COOP = Path('/content/cooperative_run.py')
COOP.write_text(subprocess.check_output(['git','-C',str(WORK),'show',TOOLS_COMMIT+':tools/cooperative_run.py'],text=True))


In [ ]:
# 3. Google Drive 연결 · 실험 선택
from google.colab import drive
from datetime import datetime, timezone
import re

drive.mount('/content/drive')
CONFIG = 'config/adaptive_extended_taste.yaml'  # 약 10~15시간: config/adaptive_reliable.yaml, 빠른 점검: config/adaptive_pilot.yaml
EXPERIMENT = 'extended_taste_20261110_001'  # 설정/소스가 달라지면 새 ID를 쓰세요.
if not re.fullmatch(r'[A-Za-z0-9_-]+',EXPERIMENT):
    raise ValueError('실험 ID에는 영문, 숫자, 밑줄, 하이픈만 사용하세요.')
OUTPUT = Path('/content/drive/MyDrive/foodcourt_adaptive_sim')/EXPERIMENT
OUTPUT.mkdir(parents=True,exist_ok=True)
from kiosk_adaptive.study import atomic_json
WORKERS = os.cpu_count() or 1  # 목적함수×반복 작업을 CPU 코어 수만큼 병렬 실행
launch = dict(source_commit=SOURCE_COMMIT,config=CONFIG,experiment=EXPERIMENT)
launch_path = OUTPUT/'launch.json'
if launch_path.exists() and json.loads(launch_path.read_text()) != launch:
    raise RuntimeError('이 폴더는 다른 소스/설정으로 생성됐습니다. 새 실험 ID를 지정하세요.')
atomic_json(launch_path,launch)
print('저장 위치:',OUTPUT)
print('병렬 작업 수:',WORKERS)
print((WORK/CONFIG).read_text())


In [ ]:
# 4-B. 여러 런타임 동시 실행 (4번 대신 실행)
# 여러 Colab 런타임이 같은 실험 폴더에서 작업을 나눠 맡습니다. 작업마다 claim.json을 남겨 중복 학습을 막고,
# 모든 목적함수의 반복 0~2를 먼저 끝낸 뒤 반복 3~4를 진행합니다. 결과는 4번과 동일합니다(작업별 시드 고정).
# 실험 폴더와 같은 torch 빌드 사용 (manifest 일치; GPU/TPU 런타임은 기본 torch가 다름)
_v=subprocess.run([sys.executable,'-c','import torch,numpy;print(torch.__version__,numpy.__version__)'],capture_output=True,text=True).stdout.strip()
print('torch/numpy:',_v)
if not _v.startswith('2.14.1+cpu'):
    subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall','--no-deps','torch==2.14.1+cpu','--index-url','https://download.pytorch.org/whl/cpu'],check=True)
    print('torch/numpy:',subprocess.run([sys.executable,'-c','import torch,numpy;print(torch.__version__,numpy.__version__)'],capture_output=True,text=True).stdout.strip())
import socket
RUNTIME = socket.gethostname()
coop_log = OUTPUT/f'run_{RUNTIME}.log'
coop_status = OUTPUT/f'run_status_{RUNTIME}.json'
def coop_state(value,**extra):
    atomic_json(coop_status,dict(status=value,runtime=RUNTIME,workers=WORKERS,updated_utc=datetime.now(timezone.utc).isoformat(),**extra))
command=[sys.executable,'-u',str(COOP),'--config',str(WORK/CONFIG),'--output',str(OUTPUT),'--workers',str(WORKERS),'--runtime',RUNTIME]
coop_state('running',command=command)
process=None
try:
    with coop_log.open('a',encoding='utf-8',buffering=1) as log:
        process=subprocess.Popen(command,cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            log.write(line)
            if ' start ' in line or ' done ' in line or 'FAILED' in line or 'aggregated' in line or 'runtime ' in line:
                print(line,end='')
        code=process.wait()
    coop_state('complete' if code==0 else 'failed',exit_code=code)
    if code:raise RuntimeError(f'실행 실패: {code}. {coop_log.name}를 확인하세요.')
except KeyboardInterrupt:
    if process is not None and process.poll() is None:
        process.terminate()
        try:process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill();process.wait()
    coop_state('interrupted')
    print('중단됨. 다시 실행하면 마지막 체크포인트부터 이어갑니다.')
